In [1]:
%pip install owlready2

Note: you may need to restart the kernel to use updated packages.


## Part1

In [2]:
from rdflib import Graph as RDFGraph
from rdflib import Namespace as RDFNamespace
from rdflib import RDF, RDFS, OWL, XSD
import os

# File path
owl_file = "family.owl"

# Verify file existence
if os.path.exists(owl_file):
    print(f"File exists: {owl_file}\n")
else:
    print(f"File does not exist: {owl_file}")
    exit()

# Use rdflib to read OWL file
g = RDFGraph()
g.parse(owl_file, format="xml")
print(f"Total triples: {len(g)}\n")

# Retrieve namespaces
namespaces = dict(g.namespaces())
print("Namespaces:\n")
for prefix, ns in namespaces.items():
    print(f"  {prefix}: {ns}")

# Main namespace (ontology namespace of family.owl)
NS = RDFNamespace("http://www.owl-ontologies.com/unnamed.owl#")
print(f"\nUsing namespace: {NS}")

# List all classes
print("\nClasses")
classes = set(g.subjects(RDF.type, OWL.Class))
for cls in sorted(classes):
    class_name = str(cls).split("#")[-1]
    print(f"  {class_name}")

# List all properties (Object Properties and Datatype Properties)
print("\nProperties")
obj_props = set(g.subjects(RDF.type, OWL.ObjectProperty))
data_props = set(g.subjects(RDF.type, OWL.DatatypeProperty))
all_props = obj_props | data_props
for prop in sorted(all_props):
    prop_name = str(prop).split("#")[-1]
    print(f"  {prop_name}")

# List all entities and their attributes
print("\nIndividual and Property")
# Get all resources that are not classes or properties
individuals = set()
for s, p, o in g:
    if (s, RDF.type, OWL.Class) not in g and (s, RDF.type, OWL.ObjectProperty) not in g and (s, RDF.type, OWL.DatatypeProperty) not in g:
        individuals.add(s)

for person in sorted(individuals):
    # Skip resources of type rdfs
    if "rdfs" in str(person) or "owl" in str(person) or "rdf" in str(person):
        continue
    
    props_dict = {}
    for prop, obj in g.predicate_objects(person):
        prop_name = str(prop).split("#")[-1] if "#" in str(prop) else str(prop).split("/")[-1]
        obj_value = str(obj)
        if prop_name not in props_dict:
            props_dict[prop_name] = []
        props_dict[prop_name].append(obj_value)
    
    if props_dict:
        person_name = str(person).split("#")[-1]
        print(f"\n{person_name}:")
        for prop, values in sorted(props_dict.items()):
            for val in values:
                print(f"  {prop}: {val}")

File exists: family.owl

Total triples: 199

Namespaces:

  brick: https://brickschema.org/schema/Brick#
  csvw: http://www.w3.org/ns/csvw#
  dc: http://purl.org/dc/elements/1.1/
  dcat: http://www.w3.org/ns/dcat#
  dcmitype: http://purl.org/dc/dcmitype/
  dcterms: http://purl.org/dc/terms/
  dcam: http://purl.org/dc/dcam/
  doap: http://usefulinc.com/ns/doap#
  foaf: http://xmlns.com/foaf/0.1/
  geo: http://www.opengis.net/ont/geosparql#
  odrl: http://www.w3.org/ns/odrl/2/
  org: http://www.w3.org/ns/org#
  prof: http://www.w3.org/ns/dx/prof/
  prov: http://www.w3.org/ns/prov#
  qb: http://purl.org/linked-data/cube#
  schema: https://schema.org/
  sh: http://www.w3.org/ns/shacl#
  skos: http://www.w3.org/2004/02/skos/core#
  sosa: http://www.w3.org/ns/sosa/
  ssn: http://www.w3.org/ns/ssn/
  time: http://www.w3.org/2006/time#
  vann: http://purl.org/vocab/vann/
  void: http://rdfs.org/ns/void#
  wgs: https://www.w3.org/2003/01/geo/wgs84_pos#
  owl: http://www.w3.org/2002/07/owl#
  rd

In [3]:
from owlready2 import *
from rdflib import Graph, Namespace, RDF

# Reading data with rdflib
g = Graph()
g.parse("family.owl", format="xml")
NS = Namespace("http://www.owl-ontologies.com/unnamed.owl#")

# Charger une ontologie avec Owlready2
onto = get_ontology("family.owl").load()

# First, collect all the people and their ages
people_data = {}
for person_uri, _, age in g.triples((None, NS.age, None)):
    name = str(g.value(person_uri, NS.name))
    age_val = int(age)
    people_data[name] = age_val

print(f"read from RDF:")
for name, age in people_data.items():
    print(f"  {name}: {age}")

with onto:
    # Create the oldPerson class
    class oldPerson(Thing):
        namespace = onto

    # Creating an individual in the ontology and setting its properties
    for name, age_val in people_data.items():
        ind = onto.Person(name)
        ind.age = age_val

print("Rule: Person(?p) ∧ age(?p, ?a) ∧ swrlb:greaterThan(?a, 60) → oldPerson(?p)")
# Run the Inference Engine
with onto:
    for ind in onto.Person.instances():
        if ind.age and ind.age > 60:
            ind.is_a.append(onto.oldPerson)
    

# View Results - Individuals aged 60 and older
print("\nElderly people over sixty years old")
old_people = list(onto.oldPerson.instances())
if old_people:
    for ind in old_people:
        age = ind.age if ind.age else None
        print(f"  {ind.name} (age: {age})")
else:
    print(" no oldPerson")

read from RDF:
  Tom: 10
  Pedro: 10
  Thomas: 40
  Alex: 25
  Michael: 5
  Claude: 5
  Peter: 70
  Marie: 69
  Sylvie: 30
  John: 45
  Chloé: 18
  Paul: 38
Rule: Person(?p) ∧ age(?p, ?a) ∧ swrlb:greaterThan(?a, 60) → oldPerson(?p)

Elderly people over sixty years old
  Peter (age: 70)
  Marie (age: 69)


## Part 2

In [6]:
from rdflib import Graph, URIRef, Literal, Namespace
MVO_NS = "http://moviekg.local/ontology/"
RDF_TYPE = URIRef("http://www.w3.org/1999/02/22-rdf-syntax-ns#type")

# Load Movie KB
g_movie = Graph()
g_movie.parse("movie_kb_final.ttl",
              format="turtle")

# Application Rule: Identify all individuals who serve as both directors and screenwriters
multitalented = set()
for movie in g_movie.subjects(RDF_TYPE, URIRef(MVO_NS + "Movie")):
    directors     = set(g_movie.objects(movie, URIRef(MVO_NS + "hasDirector")))
    screenwriters = set(g_movie.objects(movie, URIRef(MVO_NS + "hasScreenwriter")))
    overlap = directors & screenwriters
    multitalented.update(overlap)

print("Movie(?m) ∧ hasDirector(?m, ?p) ∧ hasScreenwriter(?m, ?p) → MultitalentedDirector(?p)")
print(f"MultitalentedDirector found: {len(multitalented)}")
for p in list(multitalented)[:5]:
    label = g_movie.value(p, RDFS.label)
    print(f"  {label}")

Movie(?m) ∧ hasDirector(?m, ?p) ∧ hasScreenwriter(?m, ?p) → MultitalentedDirector(?p)
MultitalentedDirector found: 41
  Hayao Miyazaki
  Paul W. S. Anderson
  Guy Ritchie
  Josh Safdie
  Fred Schepisi
